# # 🛰️ SatQuery AI — Model 2: Cross-Modal Optical + SAR Fusion VQA
#
# **Team Spectra | Smart India Hackathon 2026**
#
# This notebook fine-tunes a **Dual-Stream BLIP-2** architecture with **QLoRA**
# for Visual Question Answering on **co-registered Optical + SAR image pairs**.
# Both images are processed through a shared frozen ViT + Q-Former, their
# 32-token query outputs are **concatenated** (64 visual tokens), and fed to a
# LoRA-adapted OPT-2.7B language model to generate free-form natural-language
# answers that require cross-modal reasoning.
#
# ### Pipeline
# 1. Auto-collect ~100 co-registered Optical + SAR pairs
# 2. Generate ~500 cross-modal QA pairs requiring fusion reasoning
# 3. Build a DualStreamBLIP2 wrapper (shared ViT → concat → LoRA LM)
# 4. Fine-tune with 4-bit quantization + LoRA
# 5. Evaluate with BLEU, ROUGE-L metrics
# 6. Generate visualizations & save everything to Google Drive
#
# **Requirements:** Google Colab with T4 GPU (free tier works)


# ---
# ## 1 · Environment Setup


In [ ]:

# ── Install dependencies ─────────────────────────────────────────────────────
!pip install -q "transformers>=4.36.0" "peft>=0.7.0" "bitsandbytes>=0.41.0" \
    "accelerate>=0.25.0" "datasets>=2.16.0" evaluate nltk rouge-score \
    pillow matplotlib seaborn tqdm scipy


In [ ]:

# ── Imports ──────────────────────────────────────────────────────────────────
import os, json, random, csv, gc, warnings, time, copy
from io import BytesIO
from datetime import datetime
from collections import defaultdict
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import nltk
for _pkg in ("punkt", "wordnet", "punkt_tab"):
    nltk.download(_pkg, quiet=True)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150, "font.size": 11})
print("✅ All packages imported successfully")


In [ ]:

# ── Mount Google Drive & Configuration ───────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

CONFIG = dict(
    seed            = 42,
    drive_output    = "/content/drive/MyDrive/SatQuery_AI/Model2_CrossModal",
    data_dir        = "/content/rs_crossmodal_data",
    num_pairs       = 100,          # 100 co-registered optical+SAR pairs
    image_size      = 224,
    model_name      = "Salesforce/blip2-opt-2.7b",
    lora_rank       = 16,
    lora_alpha      = 32,
    lora_dropout    = 0.05,
    learning_rate   = 2e-4,
    num_epochs      = 8,
    batch_size      = 2,
    grad_accum_steps= 4,
    max_length      = 256,
    warmup_ratio    = 0.1,
    val_split       = 0.2,
    early_stop_patience = 2,
)

# Create output directories
for _sub in ("checkpoints", "checkpoints/processor", "results", "dataset", "report"):
    os.makedirs(f"{CONFIG['drive_output']}/{_sub}", exist_ok=True)
for _sub in ("optical", "sar", "pairs"):
    os.makedirs(f"{CONFIG['data_dir']}/{_sub}", exist_ok=True)

# Reproducibility
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  Device : {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU    : {torch.cuda.get_device_name(0)}")
    print(f"💾 VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

with open(f"{CONFIG['drive_output']}/config.json", "w") as _f:
    json.dump(CONFIG, _f, indent=2)
print(f"\n📁 Output : {CONFIG['drive_output']}")
print("✅ Setup complete")


# ---
# ## 2 · Paired Dataset Collection
#
# | Source | Modality | Details |
# |--------|----------|---------|
# | **EuroSAT** (HuggingFace) | Optical | Sentinel-2 RGB, 10 land-use classes |
# | **Synthetic SAR** (from optical) | SAR | Calibrated speckle + class-specific scattering |
#
# Each pair = (Optical, SAR) of the **same geographic scene**.


In [ ]:

# ── 2a  Download Optical Images (EuroSAT) ───────────────────────────────────
from datasets import load_dataset

print("=" * 60)
print("📡  STEP 1 : Downloading EuroSAT (Optical)")
print("=" * 60)

EUROSAT_CLASSES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway",
    "Industrial", "Pasture", "PermanentCrop", "Residential",
    "River", "SeaLake",
]

eurosat_ds = None
for _src in ("blanchon/EuroSAT", "tanganke/EuroSAT"):
    try:
        print(f"  Trying {_src} …")
        eurosat_ds = load_dataset(_src, split="train", trust_remote_code=True)
        print(f"  ✅ Loaded: {len(eurosat_ds)} images")
        break
    except Exception as _e:
        print(f"  ❌ {_e}")

# Fallback: download ZIP from Zenodo
if eurosat_ds is None:
    print("\n  Downloading from Zenodo …")
    os.system(
        "wget -q https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip "
        "-O /content/eurosat.zip && "
        "unzip -q -o /content/eurosat.zip -d /content/eurosat_raw/"
    )
    _root = Path("/content/eurosat_raw")
    _candidates = list(_root.rglob("*.jpg")) + list(_root.rglob("*.tif"))
    print(f"  Found {len(_candidates)} image files")

optical_samples = []
per_class = CONFIG["num_pairs"] // len(EUROSAT_CLASSES)  # 10 per class

if eurosat_ds is not None:
    class_counts = defaultdict(int)
    indices = list(range(len(eurosat_ds)))
    random.shuffle(indices)

    for idx in indices:
        sample = eurosat_ds[idx]
        label  = sample["label"]
        cls    = EUROSAT_CLASSES[label] if label < len(EUROSAT_CLASSES) else f"class_{label}"
        if class_counts[cls] >= per_class:
            continue
        img = sample["image"]
        if not isinstance(img, Image.Image):
            img = Image.fromarray(np.array(img))
        img = img.convert("RGB")
        _path = f"{CONFIG['data_dir']}/optical/{cls}_{class_counts[cls]:02d}.png"
        img.save(_path)
        optical_samples.append(dict(image_path=_path, cls=cls, modality="optical"))
        class_counts[cls] += 1
        if len(optical_samples) >= CONFIG["num_pairs"]:
            break
else:
    _root = Path("/content/eurosat_raw")
    for cls_dir in sorted(p for p in _root.rglob("*") if p.is_dir()):
        cls = cls_dir.name
        imgs = sorted(cls_dir.glob("*.*"))
        for i, ip in enumerate(imgs[:per_class]):
            _path = f"{CONFIG['data_dir']}/optical/{cls}_{i:02d}.png"
            Image.open(ip).convert("RGB").save(_path)
            optical_samples.append(dict(image_path=_path, cls=cls, modality="optical"))

del eurosat_ds; gc.collect()
print(f"\n  Collected {len(optical_samples)} optical images")


In [ ]:

# ── 2b  Generate Paired SAR Images ──────────────────────────────────────────
print("\n" + "=" * 60)
print("📡  STEP 2 : Generating Paired SAR Images")
print("=" * 60)
print("  Creating SAR counterpart for each optical image with:")
print("  • Class-specific radar scattering signatures")
print("  • Multiplicative speckle noise (Gamma K=4)")
print("  • Log-dB backscatter transform")
print("  ⚠️  Replace with real Sentinel-1 data for production\n")

# Class-specific SAR scattering parameters
SAR_SCATTERING = {
    # Urban/Industrial: strong double-bounce → brighter, sharper edges
    "Highway":     dict(gamma_k=3, edge_boost=1.8, base_mul=1.3, texture="sharp"),
    "Industrial":  dict(gamma_k=3, edge_boost=2.0, base_mul=1.5, texture="sharp"),
    "Residential": dict(gamma_k=3, edge_boost=1.6, base_mul=1.4, texture="sharp"),
    # Water: specular reflection → very dark, smooth
    "River":       dict(gamma_k=8, edge_boost=0.3, base_mul=0.4, texture="smooth"),
    "SeaLake":     dict(gamma_k=10, edge_boost=0.2, base_mul=0.3, texture="smooth"),
    # Forest/Vegetation: volume scattering → moderate, uniform texture
    "Forest":                dict(gamma_k=4, edge_boost=0.8, base_mul=1.0, texture="volume"),
    "HerbaceousVegetation":  dict(gamma_k=5, edge_boost=0.7, base_mul=0.9, texture="volume"),
    # Agriculture: diffuse scattering → variable texture
    "AnnualCrop":    dict(gamma_k=4, edge_boost=0.9, base_mul=1.0, texture="diffuse"),
    "Pasture":       dict(gamma_k=5, edge_boost=0.6, base_mul=0.8, texture="diffuse"),
    "PermanentCrop": dict(gamma_k=4, edge_boost=1.0, base_mul=1.1, texture="diffuse"),
}

def synthesize_sar(optical_path, cls):
    """Generate a SAR-like image from an optical image with class-specific scattering."""
    params = SAR_SCATTERING.get(cls, dict(gamma_k=4, edge_boost=1.0, base_mul=1.0, texture="diffuse"))

    arr = np.array(Image.open(optical_path).convert("L"), dtype=np.float64)

    # Apply class-specific base multiplier
    arr = arr * params["base_mul"]

    # Edge enhancement for urban (double-bounce simulation)
    if params["texture"] == "sharp":
        from scipy.ndimage import sobel
        edges = np.sqrt(sobel(arr, axis=0)**2 + sobel(arr, axis=1)**2)
        arr = arr + edges * params["edge_boost"]

    # Smooth out water surfaces (specular reflection)
    elif params["texture"] == "smooth":
        from scipy.ndimage import gaussian_filter
        arr = gaussian_filter(arr, sigma=3) * params["base_mul"]
        arr = np.clip(arr, 0, 80)  # Water is dark in SAR

    # Volume scattering for vegetation
    elif params["texture"] == "volume":
        from scipy.ndimage import uniform_filter
        arr = uniform_filter(arr, size=3) * params["base_mul"]
        arr += np.random.normal(0, 5, arr.shape)  # texture noise

    # Multiplicative speckle noise (Gamma distribution)
    speckle = np.random.gamma(shape=params["gamma_k"], scale=1.0/params["gamma_k"], size=arr.shape)
    arr = arr * speckle

    # Log-dB transform (standard SAR calibration)
    arr = np.clip(arr, 1, None)
    arr = 10.0 * np.log10(arr)

    # Normalize to 0-255
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255
    return Image.fromarray(arr.astype(np.uint8), mode="L")


paired_scenes = []  # Each entry: {optical_path, sar_path, cls}

for i, opt in enumerate(tqdm(optical_samples, desc="  Generating SAR pairs")):
    sar_path = f"{CONFIG['data_dir']}/sar/sar_pair_{i:04d}.png"
    sar_img = synthesize_sar(opt["image_path"], opt["cls"])
    sar_img.save(sar_path)

    paired_scenes.append(dict(
        optical_path=opt["image_path"],
        sar_path=sar_path,
        cls=opt["cls"],
        pair_id=i,
    ))

gc.collect()
print(f"\n📊 Created {len(paired_scenes)} co-registered Optical + SAR pairs")
print(f"   Classes : {len(set(s['cls'] for s in paired_scenes))}")


In [ ]:

# ── 2c  Generate Cross-Modal QA Pairs ───────────────────────────────────────
print("\n" + "=" * 60)
print("💬  STEP 3 : Generating Cross-Modal QA Pairs")
print("=" * 60)

CLASS_INFO = {
    "AnnualCrop": dict(
        desc="agricultural fields with annual crops showing seasonal planting patterns",
        feat=["crop rows", "planting patterns", "agricultural fields", "seasonal vegetation"],
        cat="agricultural", water=False, urban=False,
        sar_sig="moderate diffuse scattering from crop canopy with variable texture",
        opt_sig="green spectral signatures with visible row patterns and seasonal variation"),
    "Forest": dict(
        desc="dense forest cover with continuous tree canopy and woodland areas",
        feat=["tree canopy", "dense vegetation", "forest cover", "woodland areas"],
        cat="vegetation", water=False, urban=False,
        sar_sig="uniform volume scattering from multiple canopy layers with moderate backscatter",
        opt_sig="dark green reflectance indicating dense chlorophyll-rich canopy"),
    "HerbaceousVegetation": dict(
        desc="grasslands and herbaceous vegetation with open green areas",
        feat=["grassland", "natural vegetation", "herbaceous cover", "open meadows"],
        cat="vegetation", water=False, urban=False,
        sar_sig="low-to-moderate volume scattering with smooth texture from short vegetation",
        opt_sig="bright green reflectance with uniform spectral characteristics"),
    "Highway": dict(
        desc="major road infrastructure with highway corridors and transportation networks",
        feat=["highway lanes", "road infrastructure", "transportation corridors", "paved surfaces"],
        cat="infrastructure", water=False, urban=True,
        sar_sig="strong double-bounce returns from road barriers and surrounding structures",
        opt_sig="linear grey features with high reflectance from paved surfaces"),
    "Industrial": dict(
        desc="industrial zones with large buildings, warehouses, and manufacturing facilities",
        feat=["industrial buildings", "warehouses", "large structures", "factory complexes"],
        cat="urban", water=False, urban=True,
        sar_sig="very strong double-bounce and corner reflector returns from building walls and roofs",
        opt_sig="large rectangular bright features with shadow patterns from tall structures"),
    "Pasture": dict(
        desc="pastoral grasslands used for livestock grazing with open meadows",
        feat=["grazing land", "open meadows", "pastoral areas", "livestock fields"],
        cat="agricultural", water=False, urban=False,
        sar_sig="low diffuse scattering from short grass with minimal structural features",
        opt_sig="light green reflectance with smooth spectral uniformity"),
    "PermanentCrop": dict(
        desc="permanent crop plantations such as orchards, vineyards, or fruit trees",
        feat=["orchard trees", "permanent plantations", "vineyard rows", "perennial crops"],
        cat="agricultural", water=False, urban=False,
        sar_sig="moderate volume scattering with periodic texture from tree spacing patterns",
        opt_sig="ordered green patches with visible spacing between individual tree crowns"),
    "Residential": dict(
        desc="residential areas with housing developments, roads, and neighbourhood patterns",
        feat=["houses", "residential buildings", "urban roads", "neighbourhood layout"],
        cat="urban", water=False, urban=True,
        sar_sig="strong double-bounce returns from buildings with heterogeneous backscatter pattern",
        opt_sig="mixed spectral features from rooftops, roads, and garden vegetation"),
    "River": dict(
        desc="river channels and riparian zones with flowing water and bank vegetation",
        feat=["water channel", "river banks", "riparian vegetation", "flowing water"],
        cat="water", water=True, urban=False,
        sar_sig="very low specular reflection from smooth water surface with bright bank returns",
        opt_sig="dark blue-green linear features with vegetated bank boundaries"),
    "SeaLake": dict(
        desc="large water bodies including seas, lakes, coastal areas, and reservoirs",
        feat=["open water", "lake surface", "coastal areas", "water body"],
        cat="water", water=True, urban=False,
        sar_sig="extremely low specular returns from calm water producing dark uniform patches",
        opt_sig="dark blue reflectance from deep water with possible sunglint"),
}

def _info(cls):
    if cls in CLASS_INFO:
        return CLASS_INFO[cls]
    return dict(desc=cls.lower().replace("_", " "), feat=[cls.lower()],
                cat="general", water=False, urban=False,
                sar_sig="typical radar backscatter", opt_sig="standard spectral features")


def generate_crossmodal_qa(scene):
    """Return 5 cross-modal QA dicts requiring information from BOTH sensors."""
    c   = scene["cls"]
    inf = _info(c)
    qa  = []

    # Q1 – Cross-modal comparison
    qa.append(dict(
        question="Compare what the optical and SAR images reveal about this area.",
        answer=(f"The optical image shows {inf['opt_sig']}, indicating {inf['desc']}. "
                f"The SAR image complements this with {inf['sar_sig']}. "
                f"Together, both sensors confirm this is a {inf['cat']} area with "
                f"{inf['feat'][0]} as the dominant feature.")))

    # Q2 – SAR-augmented insight
    qa.append(dict(
        question="What additional information does the SAR image provide beyond the optical?",
        answer=(f"While the optical image shows surface appearance through {inf['opt_sig']}, "
                f"the SAR image adds structural information through {inf['sar_sig']}. "
                f"This radar-derived insight reveals physical properties like surface roughness "
                f"and vertical structure that optical sensors cannot capture.")))

    # Q3 – Fused land-cover identification
    qa.append(dict(
        question="Based on both optical and SAR data together, what is the land cover type?",
        answer=(f"Combining both sensors provides confident identification: the optical shows "
                f"{inf['opt_sig']}, while the SAR confirms with {inf['sar_sig']}. "
                f"This multi-modal evidence identifies the area as {inf['desc']}, "
                f"classified under the {inf['cat']} category.")))

    # Q4 – Cloud compensation / SAR-only scenario
    if inf["water"]:
        qa.append(dict(
            question="If clouds obscured the optical image, what could the SAR still determine?",
            answer=(f"SAR penetrates cloud cover, revealing {inf['sar_sig']}. "
                    f"Even without optical data, the SAR clearly identifies {inf['feat'][0]} "
                    f"through its characteristic specular reflection pattern, allowing confident "
                    f"detection of water bodies regardless of weather conditions.")))
    elif inf["urban"]:
        qa.append(dict(
            question="If clouds obscured the optical image, what could the SAR still determine?",
            answer=(f"SAR operates through clouds and reveals {inf['sar_sig']}. "
                    f"The strong double-bounce returns from building walls are unmistakable "
                    f"radar signatures of {inf['cat']} areas, allowing reliable detection of "
                    f"{inf['feat'][0]} even in persistent cloud cover.")))
    else:
        qa.append(dict(
            question="If clouds obscured the optical image, what could the SAR still determine?",
            answer=(f"SAR penetrates clouds and reveals {inf['sar_sig']}. "
                    f"This backscatter pattern is characteristic of {inf['desc']}, "
                    f"enabling the identification of {inf['feat'][0]} without requiring "
                    f"cloud-free optical conditions.")))

    # Q5 – Confidence assessment
    qa.append(dict(
        question="How confident is the land cover classification using both sensors versus one?",
        answer=(f"Using both sensors significantly increases confidence. "
                f"The optical alone provides {inf['opt_sig']}, which could be ambiguous. "
                f"Adding SAR with {inf['sar_sig']} provides independent structural confirmation. "
                f"The multi-modal agreement strongly supports classifying this as {inf['desc']}.")))

    return qa


# Build full cross-modal QA dataset
all_qa = []
for scene in tqdm(paired_scenes, desc="Generating cross-modal QA"):
    for qa in generate_crossmodal_qa(scene):
        all_qa.append(dict(
            optical_path=scene["optical_path"],
            sar_path=scene["sar_path"],
            cls=scene["cls"],
            pair_id=scene["pair_id"],
            question=qa["question"],
            answer=qa["answer"],
        ))

random.shuffle(all_qa)
_split = int(len(all_qa) * (1 - CONFIG["val_split"]))
train_data = all_qa[:_split]
val_data   = all_qa[_split:]

print(f"\n📊 Cross-Modal QA Dataset")
print(f"   Total    : {len(all_qa)}")
print(f"   Train    : {len(train_data)}")
print(f"   Val      : {len(val_data)}")
print(f"   Pairs    : {len(paired_scenes)}")

for _name, _data in [("train_qa_pairs.json", train_data), ("val_qa_pairs.json", val_data)]:
    with open(f"{CONFIG['drive_output']}/dataset/{_name}", "w") as f:
        json.dump(_data, f, indent=2)
print("💾 Saved to Drive")


In [ ]:

# ── 2d  Dataset Distribution & Sample Pairs ─────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class distribution
_cd = defaultdict(int)
for s in all_qa:
    _cd[s["cls"]] += 1
_cls = sorted(_cd); _cnt = [_cd[c] for c in _cls]
axes[0].barh(_cls, _cnt, color=sns.color_palette("husl", len(_cls)))
axes[0].set_xlabel("QA Pairs"); axes[0].set_title("Per-Class QA Count", fontweight="bold")

# Question type distribution
_qt = defaultdict(int)
for s in all_qa:
    if "Compare" in s["question"]:
        _qt["Cross-Modal\nComparison"] += 1
    elif "additional" in s["question"]:
        _qt["SAR-Augmented\nInsight"] += 1
    elif "Based on both" in s["question"]:
        _qt["Fused\nLand-Cover ID"] += 1
    elif "clouds" in s["question"]:
        _qt["Cloud\nCompensation"] += 1
    elif "confident" in s["question"]:
        _qt["Confidence\nAssessment"] += 1
axes[1].bar(_qt.keys(), _qt.values(), color=sns.color_palette("Set2", len(_qt)))
axes[1].set_ylabel("Count"); axes[1].set_title("Question Type Distribution", fontweight="bold")
plt.setp(axes[1].xaxis.get_majorticklabels(), fontsize=8)

# Train / val
axes[2].bar(["Train", "Val"], [len(train_data), len(val_data)],
            color=["#2ecc71", "#f39c12"])
axes[2].set_ylabel("QA Pairs"); axes[2].set_title("Train / Val Split", fontweight="bold")
for i, v in enumerate([len(train_data), len(val_data)]):
    axes[2].text(i, v + 3, str(v), ha="center", fontweight="bold")

plt.suptitle("Cross-Modal Dataset Overview", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/dataset_distribution.png", bbox_inches="tight")
plt.show()

# Sample paired images — Optical (top) + SAR (bottom)
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle("Sample Co-Registered Pairs — Optical (top) · SAR (bottom)",
             fontsize=14, fontweight="bold")
for i in range(5):
    p = paired_scenes[i * 2]
    axes[0, i].imshow(Image.open(p["optical_path"]))
    axes[0, i].set_title(f"{p['cls']}\n(Optical)", fontsize=9); axes[0, i].axis("off")
    axes[1, i].imshow(Image.open(p["sar_path"]).convert("L"), cmap="gray")
    axes[1, i].set_title(f"{p['cls']}\n(SAR)", fontsize=9); axes[1, i].axis("off")
axes[0, 0].set_ylabel("🛰️ Optical", fontsize=11, fontweight="bold")
axes[1, 0].set_ylabel("📡 SAR", fontsize=11, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/sample_pairs.png", bbox_inches="tight")
plt.show()
print("✅ Distribution & sample pair plots saved")


# ---
# ## 3 · Data Preprocessing & DataLoaders


In [ ]:

from transformers import Blip2Processor

print("=" * 60)
print("⚙️  STEP 4 : Data Pipeline (Dual-Image)")
print("=" * 60)

processor = Blip2Processor.from_pretrained(CONFIG["model_name"])


class CrossModalDataset(Dataset):
    """Cross-modal dataset — returns BOTH optical + SAR images per sample."""

    def __init__(self, data, processor, max_length=256):
        self.data = data
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Load both images
        opt_img = Image.open(item["optical_path"]).convert("RGB")
        sar_img = Image.open(item["sar_path"]).convert("L")
        # Convert SAR grayscale → 3-channel for ViT compatibility
        sar_rgb = Image.merge("RGB", [sar_img, sar_img, sar_img])

        q, a = item["question"], item["answer"]

        # Process both images through BLIP-2 image processor
        opt_enc = self.processor(images=opt_img, text="", return_tensors="pt")
        sar_enc = self.processor(images=sar_rgb, text="", return_tensors="pt")

        # Process text
        prompt = f"Question: {q} Answer: {a}"
        txt_enc = self.processor(
            images=opt_img, text=prompt, return_tensors="pt",
            padding="max_length", max_length=self.max_length, truncation=True,
        )

        opt_pv = opt_enc["pixel_values"].squeeze(0)
        sar_pv = sar_enc["pixel_values"].squeeze(0)
        ids    = txt_enc["input_ids"].squeeze(0)
        am     = txt_enc["attention_mask"].squeeze(0)

        # Labels: mask prompt tokens, keep only answer for loss
        labels = ids.clone()
        _prompt_only = f"Question: {q} Answer:"
        _plen = len(self.processor.tokenizer(_prompt_only, add_special_tokens=True)["input_ids"])
        labels[:_plen] = -100
        labels[am == 0] = -100

        return dict(
            optical_pixel_values=opt_pv,
            sar_pixel_values=sar_pv,
            input_ids=ids,
            attention_mask=am,
            labels=labels,
            question=q, answer=a,
            cls=item["cls"],
        )


def crossmodal_collate_fn(batch):
    return dict(
        optical_pixel_values=torch.stack([b["optical_pixel_values"] for b in batch]),
        sar_pixel_values    =torch.stack([b["sar_pixel_values"]     for b in batch]),
        input_ids           =torch.stack([b["input_ids"]            for b in batch]),
        attention_mask      =torch.stack([b["attention_mask"]       for b in batch]),
        labels              =torch.stack([b["labels"]              for b in batch]),
        questions           =[b["question"] for b in batch],
        answers             =[b["answer"]   for b in batch],
        classes             =[b["cls"]      for b in batch],
    )


train_ds = CrossModalDataset(train_data, processor, CONFIG["max_length"])
val_ds   = CrossModalDataset(val_data,   processor, CONFIG["max_length"])

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"],
                          shuffle=True,  collate_fn=crossmodal_collate_fn,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"],
                          shuffle=False, collate_fn=crossmodal_collate_fn,
                          num_workers=0, pin_memory=True)

_sb = next(iter(train_loader))
print(f"\n📊 Dual-Image Loaders ready")
print(f"   Train batches       : {len(train_loader)}")
print(f"   Val   batches       : {len(val_loader)}")
print(f"   optical_pixel_values: {_sb['optical_pixel_values'].shape}")
print(f"   sar_pixel_values    : {_sb['sar_pixel_values'].shape}")
print(f"   input_ids           : {_sb['input_ids'].shape}")
print("✅ Dual-image data pipeline ready")


# ---
# ## 4 · Model Setup — Dual-Stream BLIP-2 + QLoRA
#
# ```
#   ┌──────────┐        ┌──────────┐
#   │ Optical  │        │   SAR    │
#   │  Image   │        │  Image   │
#   └────┬─────┘        └────┬─────┘
#        │                    │
#   ┌────▼─────┐        ┌────▼─────┐
#   │  Shared  │(frozen)│  Shared  │  (same weights)
#   │   ViT    │        │   ViT    │
#   └────┬─────┘        └────┬─────┘
#        │                    │
#   ┌────▼─────┐        ┌────▼─────┐
#   │  Shared  │(frozen)│  Shared  │  (same weights)
#   │ Q-Former │        │ Q-Former │
#   └────┬─────┘        └────┬─────┘
#        │                    │
#   [32 tokens]          [32 tokens]
#        │                    │
#        └────────┬───────────┘
#                 │
#            ┌────▼────┐
#            │ CONCAT  │  → [64 visual tokens]
#            └────┬────┘
#                 │
#            ┌────▼─────┐
#            │ Language  │
#            │Projection │
#            └────┬─────┘
#                 │
#            ┌────▼────┐
#            │  OPT    │ ← LoRA adapters (trainable)
#            │  2.7B   │
#            └────┬────┘
#                 │
#            [Answer]
# ```


In [ ]:

from transformers import Blip2ForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType

print("=" * 60)
print("🧠  STEP 5 : Loading Dual-Stream BLIP-2 with QLoRA")
print("=" * 60)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("  Loading BLIP-2 base model (2-3 min) …")
base_model = Blip2ForConditionalGeneration.from_pretrained(
    CONFIG["model_name"],
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)
base_model = prepare_model_for_kbit_training(base_model)

# Freeze vision encoder + Q-Former
for p in base_model.vision_model.parameters():
    p.requires_grad = False
for p in base_model.qformer.parameters():
    p.requires_grad = False
print("  ✅ Vision encoder & Q-Former frozen")

# Apply LoRA to language model
lora_cfg = LoraConfig(
    r=CONFIG["lora_rank"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
base_model = get_peft_model(base_model, lora_cfg)

_train_p, _total_p = base_model.get_nb_trainable_parameters()
print(f"\n📊 Parameters (before dual-stream wrapper)")
print(f"   Total      : {_total_p:>12,}")
print(f"   Trainable  : {_train_p:>12,}  ({100*_train_p/_total_p:.2f}%)")


In [ ]:

# ── Dual-Stream Wrapper ─────────────────────────────────────────────────────

class DualStreamBLIP2(nn.Module):
    """
    Dual-stream wrapper for BLIP-2 that processes Optical + SAR images
    independently through the shared frozen ViT + Q-Former, concatenates
    their 32-token query outputs → 64 visual tokens, projects to LM space,
    and feeds to the LoRA-adapted OPT language model.
    """

    def __init__(self, blip2_model):
        super().__init__()
        self.blip2 = blip2_model

    def _encode_image(self, pixel_values):
        """Run a single image through ViT + Q-Former → query outputs."""
        bm = self.blip2
        if hasattr(bm, "base_model"):  # unwrap PEFT
            bm = bm.base_model.model if hasattr(bm, "base_model") else bm
        # Access underlying model components
        _m = bm
        while hasattr(_m, "model"):
            _m = _m.model

        vision_outputs = _m.vision_model(pixel_values=pixel_values, return_dict=True)
        image_embeds   = vision_outputs.last_hidden_state

        # Q-Former forward
        image_attention_mask = torch.ones(image_embeds.size()[:-1],
                                          dtype=torch.long, device=image_embeds.device)
        query_tokens = _m.query_tokens.expand(image_embeds.shape[0], -1, -1)
        query_outputs = _m.qformer(
            query_embeds=query_tokens,
            encoder_hidden_states=image_embeds,
            encoder_attention_mask=image_attention_mask,
            return_dict=True,
        )
        return query_outputs.last_hidden_state  # (batch, 32, qformer_hidden)

    def _get_lm_components(self):
        """Get language projection and language model from the BLIP-2 model."""
        _m = self.blip2
        while hasattr(_m, "model"):
            _m = _m.model
        return _m.language_projection, _m.language_model

    def forward(self, optical_pixel_values, sar_pixel_values,
                input_ids, attention_mask, labels=None):
        """
        Forward pass with dual-stream feature fusion.

        1. Encode optical → 32 query tokens
        2. Encode SAR    → 32 query tokens
        3. Concatenate   → 64 query tokens
        4. Project to LM embedding space
        5. Prepend to text embeddings → OPT language model
        """
        # Stream 1: Optical
        opt_query = self._encode_image(optical_pixel_values)  # (B, 32, H)
        # Stream 2: SAR
        sar_query = self._encode_image(sar_pixel_values)      # (B, 32, H)

        # ── CROSS-MODAL FUSION: Concatenate along sequence dimension ──
        fused_query = torch.cat([opt_query, sar_query], dim=1)  # (B, 64, H)

        # Project to language model hidden size
        language_projection, language_model = self._get_lm_components()
        language_model_inputs = language_projection(fused_query)  # (B, 64, LM_H)

        # Build attention mask for visual tokens (all ones)
        vis_attn = torch.ones(language_model_inputs.size()[:-1],
                              dtype=torch.long, device=language_model_inputs.device)

        # Get text embeddings from LM
        if hasattr(language_model, "model"):  # OPT structure
            text_embeds = language_model.model.decoder.embed_tokens(input_ids)
        else:
            text_embeds = language_model.get_input_embeddings()(input_ids)

        # Concatenate: [visual tokens (64)] + [text tokens]
        inputs_embeds  = torch.cat([language_model_inputs, text_embeds.to(language_model_inputs.dtype)], dim=1)
        full_attn_mask = torch.cat([vis_attn, attention_mask], dim=1)

        if labels is not None:
            # Pad labels with -100 for the 64 visual token positions
            vis_labels = torch.full((labels.shape[0], fused_query.shape[1]),
                                     -100, dtype=labels.dtype, device=labels.device)
            full_labels = torch.cat([vis_labels, labels], dim=1)
        else:
            full_labels = None

        outputs = language_model(
            inputs_embeds=inputs_embeds,
            attention_mask=full_attn_mask,
            labels=full_labels,
            return_dict=True,
        )
        return outputs

    @torch.no_grad()
    def generate(self, optical_pixel_values, sar_pixel_values,
                 input_ids, attention_mask, **gen_kwargs):
        """Generate text for inference with mixed-precision autocast support."""
        _device_type = "cuda" if torch.cuda.is_available() else "cpu"
        with torch.autocast(device_type=_device_type):
            opt_query = self._encode_image(optical_pixel_values)
            sar_query = self._encode_image(sar_pixel_values)
            fused_query = torch.cat([opt_query, sar_query], dim=1)

            language_projection, language_model = self._get_lm_components()
            language_model_inputs = language_projection(fused_query)

            vis_attn = torch.ones(language_model_inputs.size()[:-1],
                                  dtype=torch.long, device=language_model_inputs.device)

            if hasattr(language_model, "model"):
                text_embeds = language_model.model.decoder.embed_tokens(input_ids)
            else:
                text_embeds = language_model.get_input_embeddings()(input_ids)

            inputs_embeds  = torch.cat([language_model_inputs, text_embeds.to(language_model_inputs.dtype)], dim=1)
            full_attn_mask = torch.cat([vis_attn, attention_mask], dim=1)

            outputs = language_model.generate(
                inputs_embeds=inputs_embeds,
                attention_mask=full_attn_mask,
                **gen_kwargs,
            )
            return outputs


# Instantiate the dual-stream model
model = DualStreamBLIP2(base_model)
model = model.to(device) if not hasattr(base_model, "hf_device_map") else model

print("\n🔀 Dual-Stream BLIP-2 Architecture:")
print("   Optical → ViT → Q-Former → [32 tokens]")
print("   SAR     → ViT → Q-Former → [32 tokens]")
print("   Concatenated → [64 tokens] → Language Projection → OPT-2.7B (LoRA)")

if torch.cuda.is_available():
    print(f"\n💾 GPU Memory : {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")
print("\n✅ Dual-stream model ready")


# ---
# ## 5 · Training Loop


In [ ]:

from transformers import get_cosine_schedule_with_warmup

print("=" * 60)
print("🏋️  STEP 6 : Training (Cross-Modal Fusion)")
print("=" * 60)

# Collect only trainable params for optimizer
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=CONFIG["learning_rate"],
                              weight_decay=0.01, betas=(0.9, 0.999))

_total_steps  = len(train_loader) * CONFIG["num_epochs"] // CONFIG["grad_accum_steps"]
_warmup_steps = int(_total_steps * CONFIG["warmup_ratio"])
scheduler = get_cosine_schedule_with_warmup(optimizer, _warmup_steps, _total_steps)

scaler = torch.cuda.amp.GradScaler()
training_log = []
best_val_loss = float("inf")
patience_ctr  = 0
best_epoch    = 0
all_lrs       = []

print(f"   Epochs         : {CONFIG['num_epochs']}")
print(f"   Batch size     : {CONFIG['batch_size']}  (eff. {CONFIG['batch_size']*CONFIG['grad_accum_steps']})")
print(f"   Optim. steps   : {_total_steps}")
print(f"   Warmup steps   : {_warmup_steps}")
print(f"   Learning rate  : {CONFIG['learning_rate']}\n")

t0 = time.time()

for epoch in range(CONFIG["num_epochs"]):
    # ── Train ────────────────────────────────────────────────────
    model.train()
    _tloss, _tn = 0.0, 0
    optimizer.zero_grad()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']} [Train]")
    for step, batch in enumerate(pbar):
        with torch.cuda.amp.autocast():
            out = model(
                optical_pixel_values=batch["optical_pixel_values"].to(device),
                sar_pixel_values=batch["sar_pixel_values"].to(device),
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
                labels=batch["labels"].to(device),
            )
            loss = out.loss / CONFIG["grad_accum_steps"]
        scaler.scale(loss).backward()

        if (step + 1) % CONFIG["grad_accum_steps"] == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            all_lrs.append(scheduler.get_last_lr()[0])

        _tloss += loss.item() * CONFIG["grad_accum_steps"]
        _tn += 1
        pbar.set_postfix(loss=f"{_tloss/_tn:.4f}")

    avg_train = _tloss / _tn

    # ── Validate ─────────────────────────────────────────────────
    model.eval()
    _vloss, _vn = 0.0, 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{CONFIG['num_epochs']} [Val]"):
            with torch.cuda.amp.autocast():
                out = model(
                    optical_pixel_values=batch["optical_pixel_values"].to(device),
                    sar_pixel_values=batch["sar_pixel_values"].to(device),
                    input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device),
                    labels=batch["labels"].to(device),
                )
            _vloss += out.loss.item()
            _vn += 1
    avg_val = _vloss / _vn

    training_log.append(dict(epoch=epoch+1, train_loss=avg_train, val_loss=avg_val,
                             lr=optimizer.param_groups[0]["lr"]))
    print(f"\n  📈 Epoch {epoch+1:>2d}  train_loss={avg_train:.4f}  val_loss={avg_val:.4f}"
          f"  lr={optimizer.param_groups[0]['lr']:.2e}")

    # ── Checkpointing ─────────────────────────────────────────────
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_epoch = epoch + 1
        patience_ctr = 0
        # Save LoRA adapter weights
        base_model.save_pretrained(f"{CONFIG['drive_output']}/checkpoints")
        processor.save_pretrained(f"{CONFIG['drive_output']}/checkpoints/processor")
        print(f"  💾 Best model saved (epoch {best_epoch})")
    else:
        patience_ctr += 1
        if patience_ctr >= CONFIG["early_stop_patience"]:
            print(f"\n  ⏹️  Early stopping (patience {CONFIG['early_stop_patience']})")
            break

elapsed = time.time() - t0
print(f"\n⏱️  Training completed in {elapsed/60:.1f} min")
print(f"   Best epoch: {best_epoch}  val_loss: {best_val_loss:.4f}")

# Save training log
with open(f"{CONFIG['drive_output']}/results/training_log.json", "w") as f:
    json.dump(training_log, f, indent=2)
print("✅ Training complete")


# ---
# ## 6 · Evaluation & Metrics


In [ ]:

print("=" * 60)
print("📊  STEP 7 : Evaluation")
print("=" * 60)

# Reload best checkpoint (reuse model in memory to save VRAM & time, or load from Drive)
from peft import PeftModel
try:
    if 'model' in globals() and model is not None:
        eval_model = model
        print("  ✅ Using trained model from memory")
    else:
        _base = Blip2ForConditionalGeneration.from_pretrained(
            CONFIG["model_name"], quantization_config=bnb_config,
            device_map="auto", torch_dtype=torch.float16,
        )
        _base = PeftModel.from_pretrained(_base, f"{CONFIG['drive_output']}/checkpoints")
        eval_model = DualStreamBLIP2(_base)
        print("  ✅ Best checkpoint loaded from Drive")
except Exception as e:
    print(f"  ⚠️ Checkpoint load exception ({e}); using current weights")
    eval_model = model

eval_model.eval()

# Generate predictions on validation set
predictions = []
with torch.no_grad(), torch.cuda.amp.autocast():
    for batch in tqdm(val_loader, desc="Generating predictions"):
        # Build inference prompt (question only, no answer)
        for i in range(len(batch["questions"])):
            q = batch["questions"][i]
            prompt = f"Question: {q} Answer:"
            enc = processor.tokenizer(prompt, return_tensors="pt",
                                       padding="max_length", max_length=CONFIG["max_length"],
                                       truncation=True)

            gen = eval_model.generate(
                optical_pixel_values=batch["optical_pixel_values"][i:i+1].to(device),
                sar_pixel_values=batch["sar_pixel_values"][i:i+1].to(device),
                input_ids=enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device),
                max_new_tokens=128, do_sample=False, num_beams=3,
                repetition_penalty=1.2,
            )
            pred = processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
            if "Answer:" in pred:
                pred = pred.split("Answer:")[-1].strip()

            predictions.append(dict(
                question=q, ground_truth=batch["answers"][i],
                prediction=pred, cls=batch["classes"][i]))

print(f"  Generated {len(predictions)} predictions")

# Compute metrics
import evaluate as hf_evaluate

bleu_metric  = hf_evaluate.load("bleu")
rouge_metric = hf_evaluate.load("rouge")

refs  = [p["ground_truth"] for p in predictions]
preds = [p["prediction"] for p in predictions]

bleu1 = bleu_metric.compute(predictions=preds, references=[[r] for r in refs],
                             max_order=1)["bleu"]
bleu4 = bleu_metric.compute(predictions=preds, references=[[r] for r in refs],
                             max_order=4)["bleu"]
rouge = rouge_metric.compute(predictions=preds, references=refs)
rougeL = rouge["rougeL"]

# Per-prediction ROUGE-L for distribution analysis
_rouge_per = []
for p, r in zip(preds, refs):
    _s = rouge_metric.compute(predictions=[p], references=[r])["rougeL"]
    _rouge_per.append(_s)

metrics = {"BLEU-1": round(bleu1, 4), "BLEU-4": round(bleu4, 4),
           "ROUGE-L": round(rougeL, 4),
           "best_epoch": best_epoch, "best_val_loss": round(best_val_loss, 4),
           "training_time_min": round(elapsed / 60, 1),
           "num_pairs": len(paired_scenes), "num_qa": len(all_qa)}

print(f"\n📊 Cross-Modal Fusion Metrics")
print(f"   BLEU-1  = {metrics['BLEU-1']}")
print(f"   BLEU-4  = {metrics['BLEU-4']}")
print(f"   ROUGE-L = {metrics['ROUGE-L']}")

with open(f"{CONFIG['drive_output']}/results/metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# Save predictions CSV
with open(f"{CONFIG['drive_output']}/results/predictions.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["question", "ground_truth", "prediction", "cls", "rouge_l"])
    w.writeheader()
    for p, rl in zip(predictions, _rouge_per):
        w.writerow({**p, "rouge_l": round(rl, 4)})

print("💾 Metrics & predictions saved")


# ---
# ## 7 · Visualizations


In [ ]:

# ── 7a  Loss Curves & LR Schedule ───────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_x = [l["epoch"] for l in training_log]
ax1.plot(epochs_x, [l["train_loss"] for l in training_log], "o-", label="Train", c="#3498db", lw=2)
ax1.plot(epochs_x, [l["val_loss"] for l in training_log],   "s-", label="Val",   c="#e74c3c", lw=2)
ax1.axvline(best_epoch, ls="--", c="green", alpha=0.6, label=f"Best (ep {best_epoch})")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.set_title("Training & Validation Loss", fontweight="bold"); ax1.legend(); ax1.grid(True, alpha=0.3)

if all_lrs:
    ax2.plot(all_lrs, c="#9b59b6", lw=1.5)
    ax2.set_xlabel("Optimizer Step"); ax2.set_ylabel("Learning Rate")
    ax2.set_title("Cosine LR Schedule", fontweight="bold"); ax2.grid(True, alpha=0.3)
    ax2.ticklabel_format(style="sci", axis="y", scilimits=(0, 0))

plt.suptitle("Cross-Modal Fusion Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/loss_lr_curves.png", bbox_inches="tight")
plt.show()


In [ ]:

# ── 7b  Metric Bar Charts ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
_mnames = ["BLEU-1", "BLEU-4", "ROUGE-L"]
_mvals  = [metrics[m] for m in _mnames]
bars = ax.bar(_mnames, _mvals, color=["#3498db", "#2ecc71", "#e74c3c"], width=0.5, edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, _mvals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f"{val:.4f}", ha="center", fontweight="bold", fontsize=12)
ax.set_ylim(0, max(_mvals) * 1.3)
ax.set_ylabel("Score"); ax.set_title("Cross-Modal VQA — Evaluation Metrics", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/metrics_summary.png", bbox_inches="tight")
plt.show()


In [ ]:

# ── 7c  Paired Prediction Grid ──────────────────────────────────────────────
# Show optical+SAR pair → question → ground truth vs predicted answer
n_show = min(3, len(predictions))
fig, axes = plt.subplots(n_show, 2, figsize=(12, 5 * n_show))
if n_show == 1:
    axes = axes.reshape(1, -1)

for i in range(n_show):
    p = predictions[i]
    # Find the paired scene
    pair = None
    for s in paired_scenes:
        if s["cls"] == p["cls"]:
            pair = s; break
    if pair is None:
        pair = paired_scenes[0]

    axes[i, 0].imshow(Image.open(pair["optical_path"]))
    axes[i, 0].set_title(f"🛰️ Optical — {p['cls']}", fontsize=10, fontweight="bold")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(Image.open(pair["sar_path"]).convert("L"), cmap="gray")
    axes[i, 1].set_title(f"📡 SAR — {p['cls']}", fontsize=10, fontweight="bold")
    axes[i, 1].axis("off")

    _q = p["question"][:80]
    _gt = p["ground_truth"][:80]
    _pr = p["prediction"][:80]
    fig.text(0.5, 1.0 - (i / n_show) - 0.01,
             f"Q: {_q}…\nGT: {_gt}…\nPred: {_pr}…",
             ha="center", fontsize=8, style="italic",
             bbox=dict(boxstyle="round", facecolor="lightyellow", alpha=0.8))

plt.suptitle("Cross-Modal VQA — Sample Predictions", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/predictions_grid.png", bbox_inches="tight")
plt.show()


In [ ]:

# ── 7d  Score Distribution & Best / Worst ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(_rouge_per, bins=15, color="#3498db", edgecolor="white", alpha=0.8)
axes[0].axvline(np.mean(_rouge_per), color="red", ls="--", lw=2, label=f"Mean={np.mean(_rouge_per):.3f}")
axes[0].set_xlabel("ROUGE-L"); axes[0].set_ylabel("Count")
axes[0].set_title("Per-Sample ROUGE-L Distribution", fontweight="bold")
axes[0].legend()

# Best and worst examples
_sorted = sorted(zip(_rouge_per, predictions), key=lambda x: x[0])
_worst = _sorted[:2]
_best  = _sorted[-2:]
_examples = _worst + _best
_labels   = ["Worst 1", "Worst 2", "Best 2", "Best 1"]
_colors   = ["#e74c3c", "#e74c3c", "#2ecc71", "#2ecc71"]
axes[1].barh(_labels, [e[0] for e in _examples], color=_colors)
axes[1].set_xlabel("ROUGE-L"); axes[1].set_title("Best / Worst Predictions", fontweight="bold")
for i, (rl, p) in enumerate(_examples):
    axes[1].text(rl + 0.01, i, f"{p['cls']}: {p['question'][:30]}…", fontsize=7, va="center")

plt.suptitle("Cross-Modal VQA — Score Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/score_distribution.png", bbox_inches="tight")
plt.show()
print("✅ All visualizations saved")


# ---
# ## 8 · Model Export & Training Report


In [ ]:

report = f"""# SatQuery AI — Model 2 : Cross-Modal Fusion VQA Training Report

**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
**Team:** Spectra | SIH 2026

---

## Architecture
- **Model:** Dual-Stream BLIP-2 OPT-2.7B (4-bit quantized)
- **Innovation:** Shared ViT + Q-Former processes Optical & SAR independently,
  query outputs concatenated (32+32=64 visual tokens) → LoRA OPT
- **Fine-tuning:** QLoRA (rank {CONFIG['lora_rank']}, ~{_train_p:,} trainable params)
- **Training Time:** {elapsed/60:.1f} minutes on Colab T4

## Dataset
- **Paired Scenes:** {len(paired_scenes)} (co-registered Optical + SAR)
- **Optical Source:** EuroSAT (Sentinel-2), {len(EUROSAT_CLASSES)} classes
- **SAR Source:** Synthesized from optical (calibrated speckle + class-specific scattering)
- **Cross-Modal QA Pairs:** {len(all_qa)} ({len(train_data)} train / {len(val_data)} val)
- **Question Types:** 5 cross-modal categories requiring sensor fusion reasoning

## Results
| Metric | Score |
|--------|-------|
| BLEU-1 | {metrics['BLEU-1']} |
| BLEU-4 | {metrics['BLEU-4']} |
| ROUGE-L | {metrics['ROUGE-L']} |

- Best epoch: {best_epoch} (val_loss = {best_val_loss:.4f})

## Key Innovation
- **True feature-level sensor fusion:** Both images are encoded into separate
  query-token sequences by a shared frozen ViT + Q-Former, then concatenated
  at the feature level — the language model receives 64 fused visual tokens
  carrying complementary spectral (optical) and structural (SAR) information.
- Model generates **natural-language** answers requiring **cross-modal reasoning**.
"""

with open(f"{CONFIG['drive_output']}/report/training_report.md", "w") as f:
    f.write(report)

print("✅ Training report saved")
print("\n" + "=" * 60)
print("🎉  PIPELINE COMPLETE")
print("=" * 60)
print(f"\n📁 All outputs → {CONFIG['drive_output']}")
print(f"\n   BLEU-1  = {metrics['BLEU-1']}")
print(f"   BLEU-4  = {metrics['BLEU-4']}")
print(f"   ROUGE-L = {metrics['ROUGE-L']}")
print(f"   Best epoch {best_epoch}   val_loss={best_val_loss:.4f}")


# ---
# ## 9 · Interactive Cross-Modal Demo
#
# Test the fine-tuned model with any Optical + SAR pair + question.


In [ ]:

def satquery_crossmodal(optical_path, sar_path, question):
    """Run cross-modal VQA inference on an Optical + SAR pair."""
    opt_img = Image.open(optical_path).convert("RGB")
    sar_img = Image.open(sar_path).convert("L")
    sar_rgb = Image.merge("RGB", [sar_img, sar_img, sar_img])

    # Process images
    opt_enc = processor(images=opt_img, text="", return_tensors="pt")
    sar_enc = processor(images=sar_rgb, text="", return_tensors="pt")

    # Process prompt
    prompt = f"Question: {question} Answer:"
    txt_enc = processor.tokenizer(prompt, return_tensors="pt",
                                   padding="max_length", max_length=CONFIG["max_length"],
                                   truncation=True)

    with torch.no_grad(), torch.cuda.amp.autocast():
        gen = eval_model.generate(
            optical_pixel_values=opt_enc["pixel_values"].to(device),
            sar_pixel_values=sar_enc["pixel_values"].to(device),
            input_ids=txt_enc["input_ids"].to(device),
            attention_mask=txt_enc["attention_mask"].to(device),
            max_new_tokens=128, do_sample=False, num_beams=3,
            repetition_penalty=1.2,
        )
    ans = processor.batch_decode(gen, skip_special_tokens=True)[0].strip()
    if "Answer:" in ans:
        ans = ans.split("Answer:")[-1].strip()
    return ans


# ── Demo queries ─────────────────────────────────────────────────────────────
print("🛰️ + 📡  SatQuery AI — Cross-Modal Fusion VQA Demo")
print("=" * 55)

_demos = [
    (paired_scenes[0],  "Compare what the optical and SAR images reveal about this area."),
    (paired_scenes[5],  "What additional information does the SAR image provide beyond the optical?"),
    (paired_scenes[10], "Based on both optical and SAR data together, what is the land cover type?"),
    (paired_scenes[15], "If clouds obscured the optical image, what could the SAR still determine?"),
]

fig, axes = plt.subplots(len(_demos), 2, figsize=(10, 5 * len(_demos)))
for i, (scene, q) in enumerate(_demos):
    ans = satquery_crossmodal(scene["optical_path"], scene["sar_path"], q)

    axes[i, 0].imshow(Image.open(scene["optical_path"]))
    axes[i, 0].set_title(f"🛰️ Optical — {scene['cls']}", fontsize=10); axes[i, 0].axis("off")
    axes[i, 1].imshow(Image.open(scene["sar_path"]).convert("L"), cmap="gray")
    axes[i, 1].set_title(f"📡 SAR — {scene['cls']}", fontsize=10); axes[i, 1].axis("off")

    print(f"\n🔀 [{scene['cls']}]")
    print(f"   Q: {q}")
    print(f"   A: {ans}")

plt.suptitle("SatQuery AI — Cross-Modal Fusion VQA Demo", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CONFIG['drive_output']}/results/demo_outputs.png", bbox_inches="tight")
plt.show()

print("\n✅ Demo complete — cross-modal fusion model ready for SatQuery AI integration!")
